# Local Provider

> Local file system provider for interactive directory navigation.

In [ ]:
#| default_exp providers.local

In [ ]:
#| export
import asyncio
import os
from pathlib import Path
from typing import List, Optional, Tuple

from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_file_discovery.core.config import ExtensionMapping
from cjm_file_discovery.utils.formatting import get_mime_type

from cjm_fasthtml_file_browser.core.models import DirectoryListing

## LocalFileSystemProvider

Local file system implementation of the FileSystemProvider protocol for interactive directory browsing.

In [ ]:
#| export
class LocalFileSystemProvider:
    """Local file system provider for interactive navigation."""
    
    def __init__(
        self,
        extension_mapping: Optional[ExtensionMapping] = None  # For file type detection
    ):
        """Initialize with optional extension mapping."""
        self._extension_mapping = extension_mapping or ExtensionMapping()

    @property
    def name(self) -> str:  # Provider identifier
        """Provider identifier."""
        return "local"

    @property
    def root_path(self) -> str:  # Root path (filesystem root)
        """Root path for this provider."""
        return "/"

    @property
    def path_separator(self) -> str:  # Path separator
        """Path separator character."""
        return os.sep
    
    def _get_file_info(
        self,
        path: Path  # Path object
    ) -> Optional[FileInfo]:  # FileInfo or None on error
        """Get FileInfo for a path."""
        try:
            stat = path.stat()
            is_dir = path.is_dir()
            ext = path.suffix.lstrip('.').lower() if path.suffix and not is_dir else None
            file_type = self._extension_mapping.get_type(ext) if ext else FileType.OTHER
            
            return FileInfo(
                name=path.name,
                path=str(path.absolute()),
                is_directory=is_dir,
                size=stat.st_size if not is_dir else None,
                modified=stat.st_mtime,
                created=getattr(stat, 'st_birthtime', stat.st_ctime),
                file_type=file_type if not is_dir else FileType.OTHER,
                extension=ext,
                mime_type=get_mime_type(str(path)) if not is_dir else None,
                provider_name=self.name
            )
        except (OSError, PermissionError):
            return None

    def list_directory(
        self,
        path: str  # Directory path to list
    ) -> DirectoryListing:  # Directory contents
        """List contents of a directory."""
        p = Path(path).resolve()
        
        # Check if path exists and is a directory
        if not p.exists():
            return DirectoryListing(
                path=str(p),
                items=[],
                error=f"Path does not exist: {path}"
            )
        
        if not p.is_dir():
            return DirectoryListing(
                path=str(p),
                items=[],
                error=f"Path is not a directory: {path}"
            )
        
        # Get parent path
        parent = p.parent if p != p.parent else None
        
        # List directory contents
        items: List[FileInfo] = []
        try:
            for entry in p.iterdir():
                file_info = self._get_file_info(entry)
                if file_info is not None:
                    items.append(file_info)
        except PermissionError:
            return DirectoryListing(
                path=str(p),
                items=[],
                parent_path=str(parent) if parent else None,
                provider_name=self.name,
                error="Permission denied"
            )
        
        return DirectoryListing(
            path=str(p),
            items=items,
            parent_path=str(parent) if parent else None,
            provider_name=self.name,
            total_items=len(items)
        )

    async def list_directory_async(
        self,
        path: str  # Directory path to list
    ) -> DirectoryListing:  # Directory contents
        """Async list contents of a directory."""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.list_directory, path)

    def get_file_info(
        self,
        path: str  # Path to file/directory
    ) -> Optional[FileInfo]:  # FileInfo or None if not found
        """Get metadata for a single file/directory."""
        p = Path(path)
        if not p.exists():
            return None
        return self._get_file_info(p)

    def get_parent_path(
        self,
        path: str  # Current path
    ) -> Optional[str]:  # Parent path, or None if at root
        """Get parent directory path."""
        p = Path(path).resolve()
        parent = p.parent
        if parent == p:  # At root
            return None
        return str(parent)

    def join_path(
        self,
        base: str,   # Base path
        *parts: str  # Path parts to join
    ) -> str:  # Joined path
        """Join path components."""
        return str(Path(base).joinpath(*parts))

    def normalize_path(
        self,
        path: str  # Path to normalize
    ) -> str:  # Normalized/resolved path
        """Normalize/resolve a path."""
        return str(Path(path).resolve())

    def is_valid_path(
        self,
        path: str  # Path to validate
    ) -> Tuple[bool, Optional[str]]:  # (valid, error_message)
        """Validate path."""
        try:
            p = Path(path).resolve()
            if not p.exists():
                return (False, f"Path does not exist: {path}")
            return (True, None)
        except (ValueError, OSError) as e:
            return (False, str(e))

    def path_exists(
        self,
        path: str  # Path to check
    ) -> bool:  # True if path exists
        """Check if path exists."""
        return Path(path).exists()

    def is_directory(
        self,
        path: str  # Path to check
    ) -> bool:  # True if path is a directory
        """Check if path is a directory."""
        return Path(path).is_dir()
    
    def get_home_path(self) -> str:  # User's home directory
        """Get the user's home directory."""
        return str(Path.home())

In [ ]:
import tempfile

# Test LocalFileSystemProvider
provider = LocalFileSystemProvider()

# Test properties
assert provider.name == "local"
assert provider.root_path == "/"
assert provider.path_separator in ["/", "\\"]

# Test with a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    # Create some test files
    (Path(tmpdir) / "file1.py").write_text("print('hello')")
    (Path(tmpdir) / "file2.txt").write_text("hello world")
    (Path(tmpdir) / "subdir").mkdir()
    (Path(tmpdir) / ".hidden").write_text("hidden file")
    
    # Test list_directory
    listing = provider.list_directory(tmpdir)
    assert listing.error is None
    assert listing.path == str(Path(tmpdir).resolve())
    assert listing.parent_path is not None
    assert listing.total_items == 4  # file1.py, file2.txt, subdir, .hidden
    
    # Verify files are in listing
    names = [f.name for f in listing.items]
    assert "file1.py" in names
    assert "file2.txt" in names
    assert "subdir" in names
    
    # Test get_file_info
    py_file = Path(tmpdir) / "file1.py"
    info = provider.get_file_info(str(py_file))
    assert info is not None
    assert info.name == "file1.py"
    assert info.is_directory == False
    assert info.extension == "py"
    assert info.file_type == FileType.CODE
    
    # Test directory info
    subdir = Path(tmpdir) / "subdir"
    dir_info = provider.get_file_info(str(subdir))
    assert dir_info is not None
    assert dir_info.is_directory == True
    
    # Test get_parent_path
    parent = provider.get_parent_path(str(subdir))
    assert parent == str(Path(tmpdir).resolve())
    
    # Test root has no parent
    assert provider.get_parent_path("/") is None
    
    # Test join_path
    joined = provider.join_path(tmpdir, "subdir", "file.txt")
    assert "subdir" in joined
    assert "file.txt" in joined
    
    # Test normalize_path
    normalized = provider.normalize_path(tmpdir + "/subdir/../file1.py")
    assert ".." not in normalized
    
    # Test is_valid_path
    valid, error = provider.is_valid_path(tmpdir)
    assert valid == True
    assert error is None
    
    valid, error = provider.is_valid_path("/nonexistent/path/here")
    assert valid == False
    assert error is not None
    
    # Test path_exists
    assert provider.path_exists(tmpdir) == True
    assert provider.path_exists("/nonexistent") == False
    
    # Test is_directory
    assert provider.is_directory(tmpdir) == True
    assert provider.is_directory(str(py_file)) == False
    
    # Test get_home_path
    home = provider.get_home_path()
    assert provider.path_exists(home)
    assert provider.is_directory(home)

print("LocalFileSystemProvider tests passed!")

LocalFileSystemProvider tests passed!


In [ ]:
# Test error handling
provider = LocalFileSystemProvider()

# Test listing non-existent directory
listing = provider.list_directory("/nonexistent/path/here")
assert listing.error is not None
assert "does not exist" in listing.error
assert listing.items == []

# Test listing a file (not a directory)
with tempfile.NamedTemporaryFile() as f:
    listing = provider.list_directory(f.name)
    assert listing.error is not None
    assert "not a directory" in listing.error

# Test get_file_info for non-existent file
info = provider.get_file_info("/nonexistent/file.txt")
assert info is None

print("Error handling tests passed!")

Error handling tests passed!


In [ ]:
from cjm_fasthtml_file_browser.core.protocols import FileSystemProvider

# Verify LocalFileSystemProvider implements the protocol
provider = LocalFileSystemProvider()
assert isinstance(provider, FileSystemProvider)
print("Protocol implementation verified!")

Protocol implementation verified!


In [ ]:
# Test async method (using top-level await for Jupyter compatibility)
provider = LocalFileSystemProvider()

with tempfile.TemporaryDirectory() as tmpdir:
    (Path(tmpdir) / "test.txt").write_text("hello")
    listing = await provider.list_directory_async(tmpdir)
    assert listing.error is None
    assert len(listing.items) == 1

print("Async tests passed!")

Async tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()